<a href="https://colab.research.google.com/github/samuelezranas/morse_model/blob/main/morse_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Instalasi Library

In [2]:
!pip install pydub librosa tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl (351.2 MB)


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\samue\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python313\\site-packages\\tensorflow\\include\\external\\com_github_grpc_grpc\\src\\core\\credentials\\call\\gcp_service_account_identity\\gcp_service_account_identity_credentials.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\samue\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Generator Dataset Sintetis

In [3]:
import numpy as np
from pydub import AudioSegment
from pydub.generators import Sine
import os
import random

# Mapping Morse (Logika dari Kaggle)
MORSE_CODE_DICT = {'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.', 'F': '..-.', 'G': '--.', 'H': '....', 'I': '..', 'J': '.---', 'K': '-.-', 'L': '.-..', 'M': '--', 'N': '-.', 'O': '---', 'P': '.--.', 'Q': '--.-', 'R': '.-.', 'S': '...', 'T': '-', 'U': '..-', 'V': '...-', 'W': '.--', 'X': '-..-', 'Y': '-.--', 'Z': '--..'}

def generate_morse_audio(morse_str, filename, wpm=20, fs=16000):
    dot_duration = 1200 / wpm  # Durasi titik dalam ms
    audio = AudioSegment.silent(duration=100)

    for symbol in morse_str:
        if symbol == '.':
            audio += Sine(600).to_audio_segment(duration=dot_duration)
        elif symbol == '-':
            audio += Sine(600).to_audio_segment(duration=dot_duration * 3)
        audio += AudioSegment.silent(duration=dot_duration) # Spasi antar simbol

    audio.export(filename, format="wav")

# Membuat folder dataset
os.makedirs('dataset/dot', exist_ok=True)
os.makedirs('dataset/dash', exist_ok=True)

# Generate 500 sampel titik dan 500 sampel garis dengan sedikit variasi durasi/pitch
for i in range(500):
    generate_morse_audio('.', f'dataset/dot/dot_{i}.wav', wpm=random.randint(15, 25))
    generate_morse_audio('-', f'dataset/dash/dash_{i}.wav', wpm=random.randint(15, 25))

print("Dataset berhasil dibuat langsung di Colab!")

C:\Users\samue\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


Dataset berhasil dibuat langsung di Colab!


In [4]:
!pip install resampy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/3.1 MB ? eta -:--:--
   ------ --------------------------------- 0.5/3.1 MB 7.9 MB/s eta 0:00:01
   ---------------------------------- ----- 2.6/3.1 MB 9.4 MB/s eta 0:00:01
   ---------------------------------------- 3.1/3.1 MB 9.2 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\samue\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import librosa

def extract_features(file_path):
    audio, sample_rate = librosa.load(file_path, res_type='soxr_hq')
    mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    return np.mean(mfccs.T, axis=0)

features = []
labels = []

for label_kind in ['dot', 'dash']:
    folder = f'dataset/{label_kind}'
    for filename in os.listdir(folder):
        data = extract_features(os.path.join(folder, filename))
        features.append(data)
        labels.append(0 if label_kind == 'dot' else 1)

X = np.array(features)
y = np.array(labels)
print("Fitur berhasil diekstrak.")

Fitur berhasil diekstrak.


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Dense(128, activation='relu', input_shape=(40,)),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(2, activation='softmax') # Output: Dot atau Dash
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(X, y, epochs=20, batch_size=32, validation_split=0.2)

ModuleNotFoundError: No module named 'tensorflow.python'

In [ ]:
# Convert model ke TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Simpan file
with open('morse_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model TFLite siap diunduh untuk aplikasi Android & WearOS kamu!")

Saved artifact at '/tmp/tmpqagu077j'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  132740502015056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502016592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502018704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502017360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502017744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  132740502019280: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model TFLite siap diunduh untuk aplikasi Android & WearOS kamu!
